# Pattern #4: Multi-Agent Systems - Parallel Specialists

**From-Scratch Implementation**

## Overview

Multi-agent systems use specialized agents working in parallel:

```
Query → Router (intent + patient id) → ┌─ PolicyAgent (RAG)     ─┐
                                       ├─ LogisticsAgent (Web)  ─┤
                                       └─ BookingLookup (tool)  ─┘
                                                 ↓
                                              Judge
                                                 ↓
                                            Final Answer
```

**Router:** Identifies intent (policy / logistics / booking) and extracts patient name/phone/email when the user asks about "my appointment".

**Agents & tool:**
- **PolicyAgent**: Internal RAG for procedures, guidelines, clinical info, what to bring
- **LogisticsAgent**: Web search for hours, contacts, operational info
- **BookingLookup**: Tool that looks up patient bookings by name/phone/email from `data/bookings.json`
- **Judge**: Merges results from the selected agents/tool

**Benefits**: Specialization + Parallel Execution = Better Performance

In [2]:
import sys
from pathlib import Path
sys.path.append('..')

from utils import create_llm_provider, get_config
from rag_internal import MedicalKnowledgeRetriever, get_retriever
from tools import get_web_search_tool, get_booking_tool
import concurrent.futures
import json
import re

# Resolve path to data/medical_guides (works whether cwd is project root or scratch_demos)
_project_root = Path.cwd() if (Path.cwd() / "data" / "medical_guides").exists() else Path.cwd().parent
DOCS_DIR = _project_root / "data" / "medical_guides"


# Initialize
config = get_config()
# Use separate store for multi-agent notebook to avoid Qdrant lock conflicts
config.config["rag"]["cache_path"] = "./store_multi"
llm = create_llm_provider()
retriever = MedicalKnowledgeRetriever(docs_directory=str(DOCS_DIR))
web_search = get_web_search_tool()
booking_tool = get_booking_tool()

# Ensure documents indexed
if not retriever._is_indexed:
    print("Indexing documents...")
    stats = retriever.index_documents()
    status = stats.get("status", "unknown")
    if status == "indexed":
        print(f"Indexed {stats.get('chunk_count', 0)} chunks from {stats.get('document_count', 0)} documents")
    elif status in ("no_documents", "no_content"):
        print("No documents in data directory. Add .txt/.pdf/.docx to data/medical_guides/ and re-run.")
    else:
        print(f"Status: {status}, documents: {stats.get('document_count', 0)}")
else:
    print(f"Documents indexed: {retriever.get_stats()['total_chunks']} chunks")

print(f"\nUsing model: {config.get('model')}")
print("Multi-agent system: Router + PolicyAgent + LogisticsAgent + Booking tool + Judge")


Indexing documents...
Found 3 document(s) to index (.txt, .pdf, .docx)...
Processing appointment_preparation.txt...
Processing common_conditions.txt...
Processing general_health.txt...
Created 27 chunks. Embedding and saving to Qdrant (store/)...
  20/27 chunks
  27/27 chunks
Done in 3.7s.
Indexed 27 chunks from 3 documents

Using model: openai/gpt-4o-mini
Multi-agent system: Router + PolicyAgent + LogisticsAgent + Booking tool + Judge


### Validation (config from `config/`, Router + agents + booking tool)

In [3]:
assert config.get("model"), "config.get('model') should be set"
assert llm is not None, "create_llm_provider() should return a provider"
assert retriever._is_indexed, "Retriever should be indexed"
assert callable(web_search.search), "web_search tool ready"
assert hasattr(booking_tool, "lookup"), "booking_tool ready"
print(" Setup valid: config, LLM, retriever, web_search, and booking_tool ready.")

 Setup valid: config, LLM, retriever, web_search, and booking_tool ready.


## Multi-Agent Implementation


In [4]:
from typing import Dict, Any
import time

try:
    from IPython.display import display, Markdown
except ImportError:
    display = print
    def Markdown(s): return s

class PolicyAgent:
    """Specialist for medical policies and procedures using internal RAG."""
    
    def __init__(self, llm, retriever):
        self.llm = llm
        self.retriever = retriever
        self.name = "PolicyAgent"
    
    def process(self, query: str) -> Dict[str, Any]:
        """Process query using internal RAG."""
        context = self.retriever.get_context(query, max_k=3)
        
        system_prompt = """
You are a medical policy specialist.
Use internal knowledge to answer questions about procedures, guidelines, and clinical protocols.
Be specific and cite sources.
""".strip()
        
        prompt = f"""
Query: {query}

Internal Knowledge:
{context}

Provide a focused answer about medical policies and procedures.
""".strip()
        
        response = self.llm.generate(prompt=prompt, system_prompt=system_prompt)
        
        return {
            "agent": self.name,
            "response": response["response"],
            "tokens": response["total_tokens"],
            "latency_ms": response["latency_ms"],
            "source_type": "internal_rag"
        }


class LogisticsAgent:
    """Specialist for operational logistics using web search."""
    
    def __init__(self, llm, web_search):
        self.llm = llm
        self.web_search = web_search
        self.name = "LogisticsAgent"
    
    def process(self, query: str) -> Dict[str, Any]:
        """Process query using web search."""
        search_result = self.web_search.search(query, max_results=3)
        search_text = self.web_search.format_results(search_result)
        
        system_prompt = """
You are a healthcare logistics specialist.
Use web search results to answer questions about hours, contacts, locations, and current operations.
Include source URLs.
""".strip()
        
        prompt = f"""
Query: {query}

Web Search Results:
{search_text}

Provide a focused answer about operational logistics.
""".strip()
        
        response = self.llm.generate(prompt=prompt, system_prompt=system_prompt)
        
        return {
            "agent": self.name,
            "response": response["response"],
            "tokens": response["total_tokens"],
            "latency_ms": response["latency_ms"],
            "source_type": "web_search"
        }


class Router:
    """Identifies intent and extracts patient identifier for booking lookups."""
    
    def __init__(self, llm):
        self.llm = llm
    
    def route(self, query: str) -> Dict[str, Any]:
        """Classify intent (policy, logistics, booking) and extract patient identifier if booking-related."""
        system_prompt = """You are a router for a healthcare multi-agent system .
Classify the user's intent and extract any patient identifier they mention.

Intent:
- policy: Questions about medical procedures, guidelines, clinical protocols, what to bring for a condition, preparation (use internal knowledge/RAG).
- logistics: Questions about hospital hours, locations, contacts, operational info (use web search).
- booking: Questions about the user's own appointment(s)—when, where, with whom, status, notes (use booking lookup by patient name/phone/email).

Extract patient identifier only when booking is true: patient_name (e.g. "John Silva"), patient_phone, or patient_email from the message. Use null for missing.

Respond with ONLY a single JSON object, no markdown, no explanation. Keys: policy (boolean), logistics (boolean), booking (boolean), patient_name (string or null), patient_phone (string or null), patient_email (string or null)."""
        prompt = f"User query: {query}"
        response = self.llm.generate(prompt=prompt, system_prompt=system_prompt)
        text = (response.get("response") or "").strip()
        # Strip markdown code block if present
        if "```" in text:
            text = re.sub(r"^```(?:json)?\s*", "", text)
            text = re.sub(r"\s*```\s*$", "", text)
        try:
            out = json.loads(text)
        except json.JSONDecodeError:
            out = {"policy": True, "logistics": True, "booking": False, "patient_name": None, "patient_phone": None, "patient_email": None}
        for key in ("policy", "logistics", "booking"):
            out[key] = bool(out.get(key))
        for key in ("patient_name", "patient_phone", "patient_email"):
            v = out.get(key)
            out[key] = (v.strip() if isinstance(v, str) and v.strip() else None) or None
        return out


class Judge:
    """Merges results from multiple agents."""
    
    def __init__(self, llm):
        self.llm = llm
    
    def merge(self, query: str, agent_results: list) -> Dict[str, Any]:
        """Merge agent responses into final answer."""
        merged_info = ""
        for result in agent_results:
            merged_info += f"\n\n{result['agent']} ({result['source_type']}):\n{result['response']}"
        
        system_prompt = """
You are a judge synthesizing information from specialist agents.
Create a comprehensive, well-organized final answer.
Give priority to internal RAG for medical info, web search for operational info.
Always end with: "This is educational information; verify with the hospital / consult your clinician."
""".strip()
        
        prompt = f"""
User Query: {query}

Specialist Agent Responses:
{merged_info}

Synthesize these into a clear, comprehensive final answer.
""".strip()
        
        response = self.llm.generate(prompt=prompt, system_prompt=system_prompt)
        
        return {
            "final_answer": response["response"],
            "tokens": response["total_tokens"],
            "latency_ms": response["latency_ms"]
        }


class MultiAgentSystem:
    """Orchestrates Router, specialist agents, and booking tool."""
    
    def __init__(self, llm, retriever, web_search, booking_tool):
        self.router = Router(llm)
        self.policy_agent = PolicyAgent(llm, retriever)
        self.logistics_agent = LogisticsAgent(llm, web_search)
        self.booking_tool = booking_tool
        self.judge = Judge(llm)
    
    def run(self, query: str, parallel: bool = True, verbose: bool = True) -> Dict[str, Any]:
        """Run multi-agent system: Router → selected agents + booking (if needed) → Judge."""
        start_time = time.time()
        
        if verbose:
            display(Markdown(f"**Query:** {query}\n"))
        
        # 1. Router: intent + patient identifier
        route = self.router.route(query)
        need_policy = route.get("policy", True)
        need_logistics = route.get("logistics", True)
        need_booking = route.get("booking", False)
        if verbose:
            display(Markdown(f"**Router:** `policy={need_policy}`, `logistics={need_logistics}`, `booking={need_booking}`  \nPatient: `{route.get('patient_name') or route.get('patient_phone') or route.get('patient_email') or '—'}`\n"))
        
        agent_results = []
        
        # 2. Booking lookup (when intent is booking and we have an identifier)
        if need_booking:
            bookings = self.booking_tool.lookup(
                patient_name=route.get("patient_name"),
                patient_phone=route.get("patient_phone"),
                patient_email=route.get("patient_email"),
            )
            booking_text = self.booking_tool.format_bookings_for_agent(bookings)
            if not (route.get("patient_name") or route.get("patient_phone") or route.get("patient_email")):
                booking_text = "Booking intent detected but no patient name, phone, or email was found in the message. Please ask the user to identify themselves (e.g. 'I'm John Silva') to look up their appointment."
            agent_results.append({
                "agent": "BookingLookup",
                "response": booking_text,
                "tokens": 0,
                "latency_ms": 0,
                "source_type": "booking_lookup",
            })
            if verbose:
                display(Markdown(f"### BookingLookup (tool)\n\n{booking_text}\n"))
        
        # 3. Run Policy and/or Logistics (parallel when both needed)
        policy_result = None
        logistics_result = None
        if need_policy or need_logistics:
            if parallel and need_policy and need_logistics:
                if verbose:
                    display(Markdown("*Executing Policy + Logistics in parallel...*\n"))
                with concurrent.futures.ThreadPoolExecutor(max_workers=2) as executor:
                    future_policy = executor.submit(self.policy_agent.process, query) if need_policy else None
                    future_logistics = executor.submit(self.logistics_agent.process, query) if need_logistics else None
                    policy_result = future_policy.result() if future_policy else None
                    logistics_result = future_logistics.result() if future_logistics else None
            else:
                if verbose and (need_policy or need_logistics):
                    display(Markdown("*Executing selected agents...*\n"))
                if need_policy:
                    policy_result = self.policy_agent.process(query)
                if need_logistics:
                    logistics_result = self.logistics_agent.process(query)
            
            if policy_result:
                agent_results.append(policy_result)
                if verbose:
                    display(Markdown(f"### PolicyAgent (RAG) — {policy_result['latency_ms']}ms\n\n{policy_result['response']}\n"))
            if logistics_result:
                agent_results.append(logistics_result)
                if verbose:
                    display(Markdown(f"### LogisticsAgent (Web) — {logistics_result['latency_ms']}ms\n\n{logistics_result['response']}\n"))
        
        # If router said no policy and no logistics and no booking, run both agents as fallback
        if not agent_results:
            if verbose:
                display(Markdown("*No intent matched; running Policy + Logistics as fallback...*\n"))
            with concurrent.futures.ThreadPoolExecutor(max_workers=2) as executor:
                future_policy = executor.submit(self.policy_agent.process, query)
                future_logistics = executor.submit(self.logistics_agent.process, query)
                policy_result = future_policy.result()
                logistics_result = future_logistics.result()
            agent_results = [policy_result, logistics_result]
            if verbose:
                display(Markdown(f"*PolicyAgent* ({policy_result['latency_ms']}ms) · *LogisticsAgent* ({logistics_result['latency_ms']}ms)\n"))
        
        if verbose:
            display(Markdown("---\n### JUDGE: Merging Results\n"))
        
        judge_result = self.judge.merge(query, agent_results)
        
        if verbose:
            display(Markdown("#### Final answer\n"))
            display(Markdown(judge_result["final_answer"]))
        
        end_time = time.time()
        total_time_ms = int((end_time - start_time) * 1000)
        total_tokens = judge_result["tokens"] + sum(r.get("tokens", 0) for r in agent_results)
        
        if verbose:
            display(Markdown(f"---\n**Summary:** Router → policy={need_policy}, logistics={need_logistics}, booking={need_booking}  \nTotal tokens: {total_tokens} · Time: {total_time_ms}ms"))
        
        return {
            "query": query,
            "route": route,
            "policy_result": policy_result,
            "logistics_result": logistics_result,
            "final_answer": judge_result["final_answer"],
            "total_tokens": total_tokens,
            "total_time_ms": total_time_ms,
            "parallel": parallel,
        }

# Create system (with booking tool)
system = MultiAgentSystem(llm, retriever, web_search, booking_tool)
print("Multi-agent system initialized (Router + Policy + Logistics + Booking + Judge) ✓")


Multi-agent system initialized (Router + Policy + Logistics + Booking + Judge) ✓


## Example: Complex Multi-Domain Query (Policy + Logistics + Booking)

One query that triggers **all three**: treatment guidelines (Policy/RAG), hospital info (Logistics/web), and the user’s own appointment (Booking lookup). The Router detects all intents and the Judge merges everything into one answer.


In [5]:
# Triggers: policy (guidelines) + logistics (hospital/dermatology) + booking (extract "John Silva", lookup appointment)
query = "I'm John Silva. What are the treatment guidelines for psoriasis, which hospital in Colombo has the best dermatology department open today, and when is my appointment?"
result = system.run(query, parallel=True, verbose=True)


**Query:** I'm John Silva. What are the treatment guidelines for psoriasis, which hospital in Colombo has the best dermatology department open today, and when is my appointment?


**Router:** `policy=True`, `logistics=True`, `booking=True`  
Patient: `John Silva`


### BookingLookup (tool)

Booking 1: APT001 | Patient: John Silva | Doctor: Dr. Nimal Perera | Specialty: General Practice | Reason: Annual health checkup | Date: 2026-02-10 at 10:00 AM | Hospital: City General Hospital | Location: Building A, 2nd Floor | Status: confirmed | Notes: Fasting blood test included
Booking 2: APT005 | Patient: John Silva | Doctor: Dr. Priya Mendis | Specialty: General Practice | Reason: Skin rash consultation | Date: 2026-02-20 at 3:00 PM | Hospital: Wellness Medical Center | Location: Consultation Room 3 | Status: confirmed | Notes: Avoid applying any creams before appointment


*Executing Policy + Logistics in parallel...*


### PolicyAgent (RAG) — 12146ms

### Treatment Guidelines for Psoriasis

Psoriasis is a chronic autoimmune condition characterized by the rapid growth of skin cells, leading to scaling on the skin's surface. The treatment approach for psoriasis typically depends on the severity of the condition and may include:

1. **Topical Treatments**: 
   - **Corticosteroids**: First-line therapy for mild to moderate psoriasis.
   - **Vitamin D Analogues**: Such as calcipotriene, which help slow skin cell growth.
   - **Retinoids**: Topical retinoids like tazarotene can help reduce inflammation.
   - **Calcineurin Inhibitors**: Such as tacrolimus and pimecrolimus, especially for sensitive areas.

2. **Phototherapy**: 
   - **UVB Phototherapy**: Effective for moderate to severe psoriasis.
   - **PUVA**: A combination of psoralen and UVA light, used for more severe cases.

3. **Systemic Treatments**: 
   - **Methotrexate**: A common systemic treatment that suppresses the immune system.
   - **Biologics**: Target specific parts of the immune system (e.g., TNF-alpha inhibitors like etanercept, adalimumab, and IL-17 inhibitors like secukinumab).
   - **Oral Retinoids**: Such as acitretin for severe cases.

4. **Lifestyle Modifications**: 
   - Stress management, maintaining a healthy weight, and avoiding triggers (like smoking and excessive alcohol) can help manage symptoms.

These guidelines are based on recommendations from the American Academy of Dermatology and the National Psoriasis Foundation (source: AAD Guidelines).

### Best Dermatology Department in Colombo

While I cannot provide real-time information about specific hospitals or their current status, Colombo has several reputable hospitals with dermatology departments. Notable institutions include:

- **National Hospital of Sri Lanka (NHSL)**: Known for its comprehensive dermatology services.
- **Asiri Surgical Hospital**: Offers specialized dermatological care.
- **Nawaloka Hospital**: Provides a range of dermatology treatments.

I recommend contacting these hospitals directly to confirm their availability and services today.

### Appointment Information

Unfortunately, I do not have access to personal appointment details. Please check your email, appointment confirmation messages, or contact your healthcare provider directly for information regarding your scheduled appointment.

If you have any further questions or need assistance, feel free to ask!


### LogisticsAgent (Web) — 6722ms

Hello John Silva,

For your inquiry regarding psoriasis treatment guidelines, the American Academy of Dermatology (AAD) outlines that treatment typically includes topical agents and phototherapy, especially for mild to moderate cases. You can find more detailed guidelines on their website: [AAD Psoriasis Guidelines](https://www.aad.org/member/clinical-quality/guidelines/psoriasis).

In Colombo, the Skin Clinic in Bambalapitiya is noted for having a top dermatology department. They offer a wide range of treatments and procedures. You can learn more about their services here: [Skin Clinic Colombo](https://skinclinic.lk/doctors/).

Regarding your appointment, it is scheduled for today at 3 PM.

If you need further assistance or have additional questions, feel free to ask!


---
### JUDGE: Merging Results


#### Final answer


Hello John Silva,

Here’s the information you requested regarding psoriasis treatment guidelines, the best dermatology department in Colombo, and your appointment details.

### Treatment Guidelines for Psoriasis
Psoriasis is a chronic autoimmune condition characterized by rapid skin cell growth, leading to scaling. Treatment varies based on severity and may include:

1. **Topical Treatments**:
   - **Corticosteroids**: First-line for mild to moderate cases.
   - **Vitamin D Analogues**: Such as calcipotriene, to slow skin cell growth.
   - **Retinoids**: Topical options like tazarotene to reduce inflammation.
   - **Calcineurin Inhibitors**: Tacrolimus and pimecrolimus for sensitive areas.

2. **Phototherapy**:
   - **UVB Phototherapy**: Effective for moderate to severe psoriasis.
   - **PUVA**: Combines psoralen with UVA light for severe cases.

3. **Systemic Treatments**:
   - **Methotrexate**: Suppresses the immune system.
   - **Biologics**: Target specific immune system components (e.g., TNF-alpha inhibitors).
   - **Oral Retinoids**: Such as acitretin for severe cases.

4. **Lifestyle Modifications**: Stress management, healthy weight maintenance, and avoiding triggers like smoking and excessive alcohol can help manage symptoms.

These guidelines are based on recommendations from the American Academy of Dermatology and the National Psoriasis Foundation.

### Best Dermatology Department in Colombo
In Colombo, the **Skin Clinic in Bambalapitiya** is noted for having an excellent dermatology department, offering a wide range of treatments and procedures. Other reputable hospitals include:
- **National Hospital of Sri Lanka (NHSL)**
- **Asiri Surgical Hospital**
- **Nawaloka Hospital**

I recommend contacting these facilities directly to confirm their availability and services today.

### Appointment Information
You have an appointment scheduled for **February 20, 2026, at 3:00 PM** at the **Wellness Medical Center** for a skin rash consultation. Please remember to avoid applying any creams before your appointment.

If you have any further questions or need assistance, feel free to ask!

This is educational information; verify with the hospital / consult your clinician.

---
**Summary:** Router → policy=True, logistics=True, booking=True  
Total tokens: 3061 · Time: 22283ms

## Example Queries: Router Intent + Booking

Each example runs **separately** so you can see exactly what the Router triggers. Run the cells one at a time.

In [6]:
# --- 1. Booking-only ---
# Trigger: Router should set booking=True, extract patient_name="John Silva", run only BookingLookup (no Policy/Logistics).
query = "I'm John Silva – when is my appointment?"
result = system.run(query, parallel=True, verbose=True)

**Query:** I'm John Silva – when is my appointment?


**Router:** `policy=False`, `logistics=False`, `booking=True`  
Patient: `John Silva`


### BookingLookup (tool)

Booking 1: APT001 | Patient: John Silva | Doctor: Dr. Nimal Perera | Specialty: General Practice | Reason: Annual health checkup | Date: 2026-02-10 at 10:00 AM | Hospital: City General Hospital | Location: Building A, 2nd Floor | Status: confirmed | Notes: Fasting blood test included
Booking 2: APT005 | Patient: John Silva | Doctor: Dr. Priya Mendis | Specialty: General Practice | Reason: Skin rash consultation | Date: 2026-02-20 at 3:00 PM | Hospital: Wellness Medical Center | Location: Consultation Room 3 | Status: confirmed | Notes: Avoid applying any creams before appointment


---
### JUDGE: Merging Results


#### Final answer


Hello John Silva,

You have two upcoming appointments:

1. **Annual Health Checkup**
   - **Doctor:** Dr. Nimal Perera
   - **Date:** February 10, 2026
   - **Time:** 10:00 AM
   - **Hospital:** City General Hospital
   - **Location:** Building A, 2nd Floor
   - **Status:** Confirmed
   - **Notes:** A fasting blood test is included, so please ensure you fast before the appointment.

2. **Skin Rash Consultation**
   - **Doctor:** Dr. Priya Mendis
   - **Date:** February 20, 2026
   - **Time:** 3:00 PM
   - **Hospital:** Wellness Medical Center
   - **Location:** Consultation Room 3
   - **Status:** Confirmed
   - **Notes:** Please avoid applying any creams before this appointment.

If you have any further questions or need to make changes, please contact the respective hospitals.

This is educational information; verify with the hospital / consult your clinician.

---
**Summary:** Router → policy=False, logistics=False, booking=True  
Total tokens: 475 · Time: 4697ms

In [7]:
# --- 2. Mixed (booking + policy) ---
# Trigger: Router sets booking=True (extract "Mary Fernando") and policy=True ("what should I bring").
# Expect: BookingLookup + PolicyAgent (RAG); Judge merges appointment details + preparation advice.
query = "I'm Mary Fernando. When is my appointment and what should I bring?"
result = system.run(query, parallel=True, verbose=True)

**Query:** I'm Mary Fernando. When is my appointment and what should I bring?


**Router:** `policy=True`, `logistics=False`, `booking=True`  
Patient: `Mary Fernando`


### BookingLookup (tool)

Booking 1: APT002 | Patient: Mary Fernando | Doctor: Dr. Sunil Silva | Specialty: General Practice | Reason: Follow-up for fever and cough | Date: 2026-02-12 at 2:30 PM | Hospital: Central Medical Clinic | Location: Ground Floor, Room 5 | Status: confirmed | Notes: Bring previous test results


*Executing selected agents...*


### PolicyAgent (RAG) — 5175ms

Hello Mary Fernando,

To prepare for your upcoming appointment, please ensure you bring the following documents:

1. **Government-issued photo ID** - This is necessary for identification purposes.
2. **Health insurance card** - To verify your insurance coverage.
3. **List of current medications** - Include dosages and any supplements you are taking.
4. **Previous medical records** - If you are seeing a new doctor, this information can be crucial.
5. **Test results from other doctors** - Relevant tests can provide important context for your visit.
6. **Insurance authorization forms** - If required by your insurance provider.

Additionally, it is helpful to prepare the following medical information:

- A summary of your current symptoms, including when they started and their severity.
- Your medical history, including past illnesses, surgeries, and hospitalizations.
- Family medical history and any known allergies (medications, foods, environmental).
- Recent lifestyle changes that may be relevant to your health.

It’s also advisable to write down any questions you may have for your healthcare provider, prioritize them, and consider bringing a family member for support.

Make sure to arrive 15-20 minutes early to complete any necessary paperwork, and wear comfortable clothing that is easy to remove if needed.

If you have any specific questions about your appointment or need further assistance, feel free to reach out to the office directly.

Best of luck with your appointment!


---
### JUDGE: Merging Results


#### Final answer


Hello Mary Fernando,

Your upcoming appointment is scheduled for **February 12, 2026, at 2:30 PM** with **Dr. Sunil Silva** at the **Central Medical Clinic**, located on the **Ground Floor, Room 5**. This appointment is a follow-up for your fever and cough, and it is confirmed.

To ensure a smooth visit, please bring the following items:

1. **Government-issued photo ID** - for identification purposes.
2. **Health insurance card** - to verify your insurance coverage.
3. **List of current medications** - including dosages and any supplements you are taking.
4. **Previous medical records** - especially if you are seeing Dr. Silva for the first time.
5. **Test results from other doctors** - relevant tests can provide important context for your visit.
6. **Insurance authorization forms** - if required by your insurance provider.

Additionally, it would be beneficial to prepare the following information:

- A summary of your current symptoms, including when they started and their severity.
- Your medical history, including past illnesses, surgeries, and hospitalizations.
- Family medical history and any known allergies (medications, foods, environmental).
- Recent lifestyle changes that may be relevant to your health.

Consider writing down any questions you may have for your healthcare provider and prioritize them. It may also be helpful to bring a family member for support.

Please arrive **15-20 minutes early** to complete any necessary paperwork, and wear comfortable clothing that is easy to remove if needed.

If you have any specific questions about your appointment or need further assistance, feel free to reach out to the office directly.

This is educational information; verify with the hospital / consult your clinician.

---
**Summary:** Router → policy=True, logistics=False, booking=True  
Total tokens: 1561 · Time: 12075ms

In [8]:
# --- 3. Policy-only ---
# Trigger: Router sets policy=True, logistics=False, booking=False.
# Expect: Only PolicyAgent (RAG) runs; internal guidelines for psoriasis.
query = "What are the treatment guidelines for psoriasis?"
result = system.run(query, parallel=True, verbose=True)

**Query:** What are the treatment guidelines for psoriasis?


**Router:** `policy=True`, `logistics=False`, `booking=False`  
Patient: `—`


*Executing selected agents...*


### PolicyAgent (RAG) — 9577ms

The treatment guidelines for psoriasis are primarily based on the severity of the condition, the extent of skin involvement, and the patient's response to previous therapies. The following is a summary of the current treatment approaches as outlined by the National Psoriasis Foundation and other clinical guidelines:

### 1. **Mild Psoriasis**
For patients with mild psoriasis (less than 3% body surface area affected), first-line treatments typically include:

- **Topical Therapies**: 
  - **Corticosteroids**: These are the most commonly prescribed topical treatments and can reduce inflammation and itching.
  - **Vitamin D Analogues**: Such as calcipotriene, which can slow skin cell growth.
  - **Topical Retinoids**: Such as tazarotene, which can help normalize skin cell growth.
  - **Coal Tar**: An older treatment that can help reduce scaling and itching.

### 2. **Moderate to Severe Psoriasis**
For moderate to severe psoriasis (more than 3% body surface area affected), treatment options may include:

- **Phototherapy**: 
  - **UVB Phototherapy**: Involves exposure to ultraviolet light, which can help reduce symptoms.
  - **PUVA**: A combination of psoralen (a medication that makes the skin more sensitive to UV light) and UVA light.

- **Systemic Therapies**: 
  - **Traditional Systemics**: Such as methotrexate, cyclosporine, and acitretin, which work throughout the body to reduce inflammation and skin cell turnover.
  - **Biologics**: These are newer medications that target specific parts of the immune system. Examples include:
    - **TNF-alpha inhibitors**: Such as etanercept, infliximab, and adalimumab.
    - **IL-17 inhibitors**: Such as secukinumab and ixekizumab.
    - **IL-23 inhibitors**: Such as guselkumab and tildrakizumab.

### 3. **Lifestyle Modifications**
In addition to pharmacological treatments, lifestyle modifications are recommended to help manage psoriasis:

- **Stress Management**: Techniques such as mindfulness, yoga, and regular exercise can help reduce flare-ups.
- **Dietary Considerations**: A balanced diet rich in fruits, vegetables, and omega-3 fatty acids may help improve skin health.
- **Avoiding Triggers**: Identifying and avoiding personal triggers such as smoking, alcohol,


---
### JUDGE: Merging Results


#### Final answer


### Treatment Guidelines for Psoriasis

Psoriasis is a chronic autoimmune condition characterized by the rapid growth of skin cells, leading to scaling and inflammation. Treatment approaches vary based on the severity of the disease, the extent of skin involvement, and the patient's response to previous therapies. Below is a comprehensive overview of the treatment guidelines as outlined by the National Psoriasis Foundation and other clinical sources.

#### 1. Mild Psoriasis
For patients with mild psoriasis (affecting less than 3% of body surface area), first-line treatments typically include:

- **Topical Therapies**:
  - **Corticosteroids**: These are the most commonly prescribed topical treatments that help reduce inflammation and itching.
  - **Vitamin D Analogues**: Such as calcipotriene, which can slow skin cell growth.
  - **Topical Retinoids**: Such as tazarotene, which normalizes skin cell growth.
  - **Coal Tar**: An older treatment that can help reduce scaling and itching.

#### 2. Moderate to Severe Psoriasis
For moderate to severe psoriasis (affecting more than 3% of body surface area), treatment options may include:

- **Phototherapy**:
  - **UVB Phototherapy**: Involves exposure to ultraviolet light, which can help alleviate symptoms.
  - **PUVA**: A combination of psoralen (a medication that increases skin sensitivity to UV light) and UVA light.

- **Systemic Therapies**:
  - **Traditional Systemics**: Such as methotrexate, cyclosporine, and acitretin, which work throughout the body to reduce inflammation and skin cell turnover.
  - **Biologics**: These newer medications target specific parts of the immune system. Examples include:
    - **TNF-alpha inhibitors**: Such as etanercept, infliximab, and adalimumab.
    - **IL-17 inhibitors**: Such as secukinumab and ixekizumab.
    - **IL-23 inhibitors**: Such as guselkumab and tildrakizumab.

#### 3. Lifestyle Modifications
In addition to pharmacological treatments, lifestyle modifications are recommended to help manage psoriasis:

- **Stress Management**: Techniques such as mindfulness, yoga, and regular exercise can help reduce flare-ups.
- **Dietary Considerations**: A balanced diet rich in fruits, vegetables, and omega-3 fatty acids may improve skin health.
- **Avoiding

---
**Summary:** Router → policy=True, logistics=False, booking=False  
Total tokens: 2215 · Time: 20633ms

In [9]:
# --- 4. Logistics-only ---
# Trigger: Router sets logistics=True, policy=False, booking=False.
# Expect: Only LogisticsAgent (web search) runs; hospital/dermatology info.
query = "Which hospital in Colombo has the best dermatology department open today?"
result = system.run(query, parallel=True, verbose=True)

**Query:** Which hospital in Colombo has the best dermatology department open today?


**Router:** `policy=False`, `logistics=True`, `booking=False`  
Patient: `—`


*Executing selected agents...*


### LogisticsAgent (Web) — 5818ms

In Colombo, two notable hospitals with strong dermatology departments are currently open today:

1. **Colombo South Teaching Hospital**
   - **Location**: Kalubowila, Dehiwala
   - **Hours**: Open 24 hours
   - **Services**: Offers a comprehensive range of dermatological services for skin conditions.
   - **More Information**: [Colombo South Teaching Hospital Dermatology Service](https://www.csth.health.gov.lk/dermatology-service)

2. **Skin Clinic in Bambalapitiya**
   - **Location**: Bambalapitiya, Colombo
   - **Hours**: Typically operates during standard business hours (specific hours not listed, so it's advisable to call ahead).
   - **Services**: Provides a variety of dermatological treatments including cosmetic procedures.
   - **More Information**: [Skin Clinic](https://skinclinic.lk/doctors/)

Both facilities are well-regarded for their dermatology services, so you can choose based on your specific needs or proximity.


---
### JUDGE: Merging Results


#### Final answer


In Colombo, two hospitals with reputable dermatology departments that are open today are:

1. **Colombo South Teaching Hospital**
   - **Location**: Kalubowila, Dehiwala
   - **Hours**: Open 24 hours
   - **Services**: This hospital offers a comprehensive range of dermatological services for various skin conditions. You can find more information about their dermatology services [here](https://www.csth.health.gov.lk/dermatology-service).

2. **Skin Clinic**
   - **Location**: Bambalapitiya, Colombo
   - **Hours**: Typically operates during standard business hours (specific hours not listed, so it's advisable to call ahead).
   - **Services**: The Skin Clinic provides a variety of dermatological treatments, including cosmetic procedures. More details can be found on their website [here](https://skinclinic.lk/doctors/).

Both facilities are well-regarded for their dermatology services, allowing you to choose based on your specific needs or proximity.

This is educational information; verify with the hospital / consult your clinician.

---
**Summary:** Router → policy=False, logistics=True, booking=False  
Total tokens: 1170 · Time: 18151ms

## Pattern Summary

Multi-agent systems leverage **specialization** and **parallel execution**:

**Architecture:**
```
Query
  ↓
Router (implicit via parallel execution)
  ├─→ PolicyAgent (RAG) ─┐
  └─→ LogisticsAgent (Web)─┤
                           ↓
                         Judge
                           ↓
                      Final Answer
```

**Key Features:**
- **Specialization**: Each agent has specific expertise
- **Parallel Execution**: Agents work simultaneously
- **Intelligent Merging**: Judge synthesizes results appropriately

**Benefits:**
- **Speed**: Parallel > Sequential (lower latency)
- **Quality**: Specialists > Generalists (better focus)
- **Scalability**: Easy to add more specialist agents
- **Clear Separation**: Internal (RAG) vs External (Web) expertise

**When to use:**
- Complex queries spanning multiple domains
- Need for both policy and operational info
- Speed is important (parallel execution helps)
- Clear specialization boundaries exist

**Trade-offs:**
- More complex architecture
- Need orchestration (router + judge)
- Higher token cost (multiple agents + judge)
- Potential for conflicting information (judge must resolve)

**Comparison with ReAct:**
- ReAct: Sequential, adaptive, visible reasoning
- Multi-Agent: Parallel, specialized, distributed processing

Multi-agent systems scale well to complex real-world applications!
